# 00 · Colab ortamı ve veri sınıfı (rejim A, B-fiz)

Bu not defteri her oturumun başında çalıştırılan kurulumdur:
GPU kontrolü → proje klasörü → `stimuli.check_all()` → veri sınıfı kontrolleri → örnek görseller → hız testi.

**Önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*.

In [ ]:
import torch, platform
print("PyTorch", torch.__version__, "| Python", platform.python_version())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU YOK → Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU (veri kontrolleri yine de çalışır)")

## 1. Proje klasörü
`GITHUB_URL`'yi ortak reponuzun adresiyle değiştirin. Repo private ise token Colab Secrets'tan (`GH_TOKEN`) okunur; public ise hiçbir şey yapmanız gerekmez. Sonuçlar her iki durumda da Drive'a yazılır (Colab kapanınca kaybolmasın diye).

In [ ]:
KAYNAK = "github"                                          # "github" ya da "drive"
GITHUB_URL = "https://github.com/KULLANICI/ebbinghaus-human-ai.git"      # ortak reponuzun adresi
DAL = "main"                                              # PR birleşmeden test için: "veri-sinifi"
DRIVE_PROJE = "/content/drive/MyDrive/ebbinghaus-proje"   # yalnızca KAYNAK = "drive" ise

import os, sys
from google.colab import drive, userdata
drive.mount("/content/drive")

if KAYNAK == "github":
    # Repo private ise: sol menü 🔑 Secrets → GH_TOKEN adıyla token ekleyin.
    # Public ise token gerekmez, bu adım atlanır.
    try:
        token = userdata.get("GH_TOKEN")
        url = GITHUB_URL.replace("https://", f"https://{token}@")
    except Exception:
        url = GITHUB_URL
    PROJE = "/content/" + GITHUB_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(PROJE):
        !git clone -q -b {DAL} {url} {PROJE}
    else:
        !git -C {PROJE} fetch -q origin {DAL} && git -C {PROJE} checkout -q {DAL} && git -C {PROJE} pull -q origin {DAL}
    SONUC = "/content/drive/MyDrive/ebbinghaus-sonuclar/results"
else:
    PROJE = DRIVE_PROJE
    SONUC = os.path.join(PROJE, "results")

os.makedirs(SONUC, exist_ok=True)
os.chdir(PROJE)
if PROJE not in sys.path:
    sys.path.insert(0, PROJE)
print("Proje :", PROJE, "→", sorted(os.listdir(PROJE)))
print("Sonuç :", SONUC)

## 2. Ortak uyaran üreteci (Kişi 1'in `stimuli.py`'si)

In [ ]:
import stimuli
print("MODEL ayarları:", stimuli.MODEL)
for (kurulum, kosul), e in stimuli.check_all().items():
    print(f"{kurulum:6s} {kosul:10s} dış sınır {e:6.1f} px")
print("stimuli.check_all ✓ — tüm koşullar tuval içinde")

## 3. Veri sınıfı kontrolleri
1. Eğitim görselleri test görselleriyle **piksel piksel aynı** yöntemle çiziliyor mu?
2. 8 test koşulunun çevre parametreleri B-fiz eğitim aralığının içinde mi?
3. B-fiz'de etiket (merkez çapı) ile çevre parametreleri arasında korelasyon ≈ 0 mı?

In [ ]:
from model import data
data.check_all()

## 4. Örnek görseller (göz kontrolü)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 8, figsize=(16, 6.6))
for row, reg in enumerate(["A", "B-fiz"]):
    ds = data.EbbinghausDataset(reg, "val", n=8, seed=0)
    for i in range(8):
        x, y = ds[i]
        axes[row, i].imshow(x[0], cmap="gray_r", vmin=0, vmax=1)
        axes[row, i].set_title(f"{reg}  y={float(y):.2f}", fontsize=9)
for i, (name, r, g) in enumerate(stimuli.condition_list()):
    img = stimuli.draw_config(canvas=data.CANVAS, center_d=data.D, ratio=r, gap=g, ref_d=data.D)
    axes[2, i].imshow(data.to_array(img), cmap="gray_r", vmin=0, vmax=1)
    axes[2, i].set_title(f"TEST {name}", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
for ax, t in zip(axes[:, 0], ["Rejim A", "Rejim B-fiz", "Test koşulları"]):
    ax.set_ylabel(t, fontsize=11)
plt.tight_layout(); plt.show()

## 5. DataLoader hız testi

In [ ]:
import time
from torch.utils.data import DataLoader
for reg in ["A", "B-fiz"]:
    ds = data.EbbinghausDataset(reg, "train", n=2000, seed=0)
    dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2)
    t = time.time()
    for x, y in dl:
        pass
    s = time.time() - t
    print(f"{reg:6s}: 2.000 görsel {s:.1f} s → 20.000'lik epoch ≈ {s*10:.0f} s"
          f" | x {tuple(x.shape)} {x.dtype} | y [{float(y.min()):.2f}, {float(y.max()):.2f}]")

## Gözden geçiren (Kişi 2) için kontrol listesi — yaklaşık 15 dk
- [ ] Hücre 2–5 hatasız çalıştı, üç kontrolde de ✓ var.
- [ ] Rejim A'da yalnız merkez daire var; B-fiz'de yaklaşık her 5 görselden 1'i tek daire.
- [ ] B-fiz çevreleri merkezden bağımsız görünüyor (büyük merkez ≠ büyük/yakın çevre).
- [ ] Test satırındaki 8 koşul, PDF'teki Şekil 1 ile aynı düzende.
- [ ] Epoch süresi tahmini 1 dakikanın altında.
- [ ] `model/data.py` başındaki aralıklar (CENTER_RANGE, SURR_D_RANGE, INNER_RANGE, N_RANGE) plana uygun.